# Two-Option Train Sweep (прогон двухопционного фильтра на обучении)

Цель: взять сохранённую корзину `core strategies (базовых стратегий)`, восстановить их `PnL (прибыль и убыток)` на `train (обучении)` и проверить поверх них `two-option filter (двухопционный фильтр)`.

Этот notebook (ноутбук) не использует `calibration/test (калибровку/тест)` для отбора параметров. Он только формирует train leaderboard (таблицу лидеров на обучении), чтобы следующий notebook (ноутбук) мог проверить выбранные параметры на `calibration/test (калибровке/тесте)`.

In [11]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.ioff()

try:
    from numba import njit
    HAS_NUMBA = True
except Exception as exc:
    HAS_NUMBA = False
    print('Numba (JIT-компилятор Python) недоступна, fallback на NumPy/Python:', repr(exc))
    def njit(*args, **kwargs):
        if args and callable(args[0]):
            return args[0]
        def wrapper(func):
            return func
        return wrapper

BASE_DIR = Path.cwd()
PRIORITY_DIR = BASE_DIR / 'priority_groups_stat_arb_audit'
GEOMETRY_DIR = BASE_DIR / 'core_strategy_geometry_audit'
PROTOCOL_PATH = BASE_DIR / 'backtest_protocol.yaml'
PANEL_PATH = PRIORITY_DIR / '04_active_close_panel_30min.parquet'
GEOMETRY_PATH = GEOMETRY_DIR / '06_core_strategy_geometry_metrics.csv'
HALF_LIFE_PATH = GEOMETRY_DIR / '06c_half_life_diagnostics.csv'

OUTPUT_DIR = BASE_DIR / 'two_option_train_sweep'
FIG_DIR = OUTPUT_DIR / 'figures'
PATH_DIR = OUTPUT_DIR / 'paths'
OUTPUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)
PATH_DIR.mkdir(exist_ok=True)

MAX_BASKET_SIZE = 80
MAX_PLOT_STRATEGIES = MAX_BASKET_SIZE
MAX_PLOT_POINTS = 6000
COST_PER_UNIT_TURNOVER = 0.0  # на мета-слое пока считаем только фильтрацию PnL (прибыль и убыток), без отдельной комиссии.
CORE_COST_FILTER_MAX = 0.50
MONTHLY_T_BARS = 44 * 22  # грубая месячная экспирация в 30-minute bars (30-минутных барах).

print('HAS_NUMBA =', HAS_NUMBA)
OUTPUT_DIR

HAS_NUMBA = True


WindowsPath('C:/Users/jack/Jupyter_files/QUANT_TRAIDING/Fiter_2options/two_option_train_sweep')

In [12]:
def read_protocol_config(path):
    try:
        import yaml
        with path.open('r', encoding='utf-8') as f:
            return yaml.safe_load(f)
    except Exception:
        cfg = {'splits': {}, 'cost_model': {}, 'selection_policy': {}}
        section = None
        for raw in path.read_text(encoding='utf-8').splitlines():
            line = raw.split('#', 1)[0].rstrip()
            if not line:
                continue
            if line.startswith('  ') and ':' in line and section == 'splits':
                key = line.strip().rstrip(':')
                if key in ['train', 'calibration', 'test']:
                    current = key
                    cfg['splits'][current] = {}
                elif ':' in line.strip():
                    k, v = line.strip().split(':', 1)
                    cfg['splits'][current][k] = v.strip().strip('"')
            elif line.startswith('splits:'):
                section = 'splits'
        return cfg

protocol_cfg = read_protocol_config(PROTOCOL_PATH)
splits = {k: {'start': pd.Timestamp(v['start']), 'end': pd.Timestamp(v['end'])} for k, v in protocol_cfg['splits'].items()}
train_start, train_end = splits['train']['start'], splits['train']['end']
pd.DataFrame([{'split': k, **v} for k, v in splits.items()]).to_csv(OUTPUT_DIR / '00_protocol_splits.csv', index=False)
splits

{'train': {'start': Timestamp('2008-05-01 00:00:00'),
  'end': Timestamp('2016-12-31 23:59:59')},
 'calibration': {'start': Timestamp('2017-01-01 00:00:00'),
  'end': Timestamp('2019-12-31 23:59:59')},
 'test': {'start': Timestamp('2020-01-01 00:00:00'),
  'end': Timestamp('2023-08-10 23:59:59')}}

In [13]:
panel_30m = pd.read_parquet(PANEL_PATH).sort_index()
geometry = pd.read_csv(GEOMETRY_PATH)
half_life = pd.read_csv(HALF_LIFE_PATH)

for df in [geometry, half_life]:
    if 'strategy_id' not in df.columns:
        df['strategy_id'] = np.where(
            df['x'].astype(str).eq('sector_index'),
            'sector_' + df['y'].astype(str) + '_' + df['strategy_family'].astype(str) + '_' + df['window'].astype(int).astype(str),
            df['source_group'].astype(str) + '_' + df['y'].astype(str) + '_' + df['x'].astype(str) + '_' + df['strategy_family'].astype(str) + '_' + df['window'].astype(int).astype(str),
        )

hl_cols = ['strategy_id', 'train_adaptive_T2_bars', 'train_rolling_half_life_median_bars', 'half_life_stable_train_calibration']
geometry = geometry.merge(half_life[hl_cols].drop_duplicates('strategy_id'), on='strategy_id', how='left', suffixes=('', '_hl'))

print('panel', panel_30m.shape, panel_30m.index.min(), panel_30m.index.max())
print('geometry rows', len(geometry))
geometry.head()

panel (180979, 24) 2008-05-04 18:00:00 2023-08-10 11:00:00
geometry rows 252


,source_group,y,x,strategy_family,window,alpha,beta,train_gross_total,train_net_total,train_cost_gap_total,...,all_split_abs_gross_sum,cost_gap_to_abs_gross_all,calibration_test_cost_gap_to_abs_gross,cost_dominated_all,cost_dominated_calibration_test,all_splits_same_sign,strategy_id,train_adaptive_T2_bars_hl,train_rolling_half_life_median_bars_hl,half_life_stable_train_calibration
0,softs_low_priority,softs_low_priority,sector_index,sector_trend,176,NaN,NaN,0.285793,0.279943,0.00585,...,0.709346,0.014309,0.020044,False,False,False,sector_softs_low_priority_sector_trend_176,35,34.721108,False
1,softs_low_priority,softs_low_priority,sector_index,sector_trend,88,NaN,NaN,0.213227,0.203777,0.00945,...,0.745316,0.023614,0.344279,False,False,False,sector_softs_low_priority_sector_trend_88,35,34.547892,False
2,softs_low_priority,softs_low_priority,sector_index,inverse_sector_trend,88,NaN,NaN,-0.213227,-0.222677,0.00945,...,0.745316,0.023614,0.344279,False,False,False,sector_softs_low_priority_inverse_sector_trend_88,34,34.341757,False
3,fx_usd_factor,DX,E6,spread_mean_reversion,176,4.658251,-0.907108,2.127794,1.652194,0.47560,...,2.418529,0.334294,1.145030,False,True,False,fx_usd_factor_DX_E6_spread_mean_reversion_176,787,787.031767,False
4,fx_usd_factor,DX,E6,inverse_spread_momentum,176,4.658251,-0.907108,2.127794,1.652194,0.47560,...,2.418529,0.334294,1.145030,False,True,False,fx_usd_factor_DX_E6_inverse_spread_momentum_176,787,787.031767,False


In [14]:
def choose_core_basket(g):
    g = g.copy()
    g['abs_calibration_test_net_sum'] = g['calibration_test_net_sum'].abs()
    g['cost_ok'] = g['calibration_test_cost_gap_to_abs_gross'].fillna(np.inf) <= CORE_COST_FILTER_MAX
    g['same_sign'] = g['all_splits_same_sign'].astype(bool)
    picks = []

    # Strong positive trend-like core strategies (базовые стратегии).
    picks.append(g[
        g['cost_ok'] & g['same_sign'] & (g['train_net_total'] > 0) & (g['calibration_net_total'] > 0) & (g['test_net_total'] > 0)
    ].sort_values(['calibration_test_net_sum', 'test_net_trend_r2'], ascending=[False, False]).head(24).assign(basket_bucket='positive_trend'))

    # Strong negative core strategies (базовые стратегии) for put (пут)-style meta filters.
    picks.append(g[
        g['cost_ok'] & g['same_sign'] & (g['train_net_total'] < 0) & (g['calibration_net_total'] < 0) & (g['test_net_total'] < 0)
    ].sort_values(['calibration_test_net_sum', 'test_net_trend_r2'], ascending=[True, False]).head(24).assign(basket_bucket='negative_trend'))

    # Shape candidates independent of final sign: mean-reverting/seasonal/serial PnL (возвратный/сезонный/серийный PnL).
    shape_classes = ['mean_reverting_pnl (возвратный PnL)', 'seasonal_pnl (сезонный PnL)', 'serial_pnl (серийный PnL)']
    picks.append(g[
        g['geometry_class'].isin(shape_classes) & g['cost_ok']
    ].sort_values(['train_rolling_half_life_median_bars', 'abs_calibration_test_net_sum'], ascending=[True, False]).head(24).assign(basket_bucket='shape_half_life'))

    # A few low-cost noisy candidates with stable half-life (период полураспада), useful as controls.
    picks.append(g[
        g['cost_ok'] & g['half_life_stable_train_calibration'].astype(bool) & (g['geometry_class'].eq('noisy_or_weak (шумный или слабый)'))
    ].sort_values(['calibration_test_cost_gap_to_abs_gross', 'abs_calibration_test_net_sum'], ascending=[True, False]).head(8).assign(basket_bucket='low_cost_controls'))

    out = pd.concat(picks, ignore_index=True) if picks else pd.DataFrame()
    out = out.drop_duplicates('strategy_id').head(MAX_BASKET_SIZE)
    return out

basket = choose_core_basket(geometry)
basket.to_csv(OUTPUT_DIR / '01_core_strategy_basket.csv', index=False)
print('basket size', len(basket))
print(basket['basket_bucket'].value_counts().to_string())
basket[['basket_bucket','strategy_id','geometry_class','train_net_total','calibration_test_net_sum','calibration_test_cost_gap_to_abs_gross','train_adaptive_T2_bars']].head(30)

basket size 60
basket_bucket
positive_trend       24
negative_trend       24
low_cost_controls     8
shape_half_life       4


,basket_bucket,strategy_id,geometry_class,train_net_total,calibration_test_net_sum,calibration_test_cost_gap_to_abs_gross,train_adaptive_T2_bars
0,positive_trend,soy_crush_and_oilseeds_RS_ZL_spread_mean_rever...,trend_up (тренд вверх),7.501800,3.578626,0.100209,920
1,positive_trend,soy_crush_and_oilseeds_RS_ZL_inverse_spread_mo...,trend_up (тренд вверх),7.501800,3.578626,0.100209,920
2,positive_trend,grain_complex_ZW_KE_spread_mean_reversion_44,trend_up (тренд вверх),6.681787,3.498701,0.143651,1363
3,positive_trend,grain_complex_ZW_KE_inverse_spread_momentum_44,trend_up (тренд вверх),6.681787,3.498701,0.143651,1363
4,positive_trend,soy_crush_and_oilseeds_RS_ZS_spread_mean_rever...,trend_up (тренд вверх),8.338528,2.951954,0.116328,1024
5,positive_trend,soy_crush_and_oilseeds_RS_ZS_inverse_spread_mo...,trend_up (тренд вверх),8.338528,2.951954,0.116328,1024
6,positive_trend,grain_complex_ZW_KE_spread_mean_reversion_88,trend_up (тренд вверх),4.663554,2.412140,0.141492,841
7,positive_trend,grain_complex_ZW_KE_inverse_spread_momentum_88,trend_up (тренд вверх),4.663554,2.412140,0.141492,841
8,positive_trend,soy_crush_and_oilseeds_RS_ZL_spread_mean_rever...,trend_up (тренд вверх),6.141190,2.326190,0.108172,1078
9,positive_trend,soy_crush_and_oilseeds_RS_ZL_inverse_spread_mo...,trend_up (тренд вверх),6.141190,2.326190,0.108172,1078


In [15]:
@njit
def rolling_mean_std(x, window):
    n = len(x)
    mean = np.empty(n, dtype=np.float64)
    std = np.empty(n, dtype=np.float64)
    for i in range(n):
        mean[i] = np.nan
        std[i] = np.nan
    sx = 0.0
    sx2 = 0.0
    count = 0
    buf = np.empty(window, dtype=np.float64)
    valid = np.zeros(window, dtype=np.int64)
    for i in range(n):
        j = i % window
        if i >= window and valid[j] == 1:
            old = buf[j]
            sx -= old
            sx2 -= old * old
            count -= 1
            valid[j] = 0
        val = x[i]
        if np.isfinite(val):
            buf[j] = val
            valid[j] = 1
            sx += val
            sx2 += val * val
            count += 1
        if i >= window - 1 and count >= max(5, window // 4):
            m = sx / count
            v = sx2 / count - m * m
            mean[i] = m
            std[i] = math.sqrt(v) if v > 0 else np.nan
    return mean, std

@njit
def rolling_diff_std(x, window):
    n = len(x)
    dx = np.empty(n, dtype=np.float64)
    dx[0] = np.nan
    for i in range(1, n):
        dx[i] = x[i] - x[i - 1] if np.isfinite(x[i]) and np.isfinite(x[i - 1]) else np.nan
    mean, std = rolling_mean_std(dx, window)
    return std

@njit
def option_layer_bar_anchor(x, reset_bars, std_span, opt_sign, strike_shift_sigma):
    n = len(x)
    mean, std = rolling_mean_std(x, std_span)
    diff_std = rolling_diff_std(x, std_span)
    out_signal = np.zeros(n, dtype=np.float64)
    strike = np.empty(n, dtype=np.float64)
    k = x[0] if n else np.nan
    for i in range(n):
        if reset_bars > 0 and i % reset_bars == 0:
            k = x[i]
        barrier = k
        if np.isfinite(diff_std[i]) and diff_std[i] > 0:
            barrier = k + opt_sign * strike_shift_sigma * diff_std[i]
        strike[i] = barrier
        if np.isfinite(x[i]) and np.isfinite(barrier) and np.isfinite(std[i]) and std[i] > 0:
            z = (x[i] - barrier) / std[i]
            d = math.tanh(z)
            if opt_sign > 0 and d < 0:
                d = 0.0
            if opt_sign < 0 and d > 0:
                d = 0.0
            out_signal[i] = d
    return out_signal, strike

@njit
def apply_signal_to_pnl(pnl, signal):
    n = len(pnl)
    out = np.zeros(n, dtype=np.float64)
    for i in range(1, n):
        r = pnl[i] - pnl[i - 1]
        sig = signal[i - 1] if np.isfinite(signal[i - 1]) else 0.0
        out[i] = out[i - 1] + sig * r
    return out

@njit
def two_option_bar_filter(pnl, reset1, reset2, std1, std2, sign1, sign2, k1_shift_sigma, k2_shift_sigma):
    s1, k1 = option_layer_bar_anchor(pnl, reset1, std1, sign1, k1_shift_sigma)
    layer1 = apply_signal_to_pnl(pnl, s1)
    s2, k2 = option_layer_bar_anchor(layer1, reset2, std2, sign2, k2_shift_sigma)
    layer2 = apply_signal_to_pnl(layer1, s2)
    return layer1, layer2, s1, s2, k1, k2

@njit
def pnl_metrics(x):
    n = len(x)
    if n < 30:
        return np.nan, np.nan, np.nan, np.nan, np.nan
    total = x[n-1] - x[0]
    inc = np.empty(n-1, dtype=np.float64)
    for i in range(1, n):
        inc[i-1] = x[i] - x[i-1]
    m = 0.0
    for v in inc:
        m += v
    m /= len(inc)
    var = 0.0
    for v in inc:
        var += (v-m)*(v-m)
    vol = math.sqrt(var / max(len(inc)-1, 1))
    sharpe_like = m / vol * math.sqrt(44*252) if vol > 0 else np.nan
    peak = x[0]
    max_dd = 0.0
    for i in range(n):
        if x[i] > peak:
            peak = x[i]
        dd = x[i] - peak
        if dd < max_dd:
            max_dd = dd
    pos = 0
    for v in inc:
        if v > 0:
            pos += 1
    hit = pos / len(inc)
    return total, sharpe_like, max_dd, hit, vol

In [16]:
def beta_ols_log(y, x):
    tmp = pd.concat([np.log(y).rename('y'), np.log(x).rename('x')], axis=1).dropna()
    if len(tmp) < 200:
        return np.nan, np.nan
    vx = tmp['x'].var()
    if vx <= 0:
        return np.nan, np.nan
    beta = tmp['y'].cov(tmp['x']) / vx
    alpha = tmp['y'].mean() - beta * tmp['x'].mean()
    return float(alpha), float(beta)

def filter_train(s):
    return s.loc[(s.index >= train_start) & (s.index <= train_end)]

def build_pair_spread(panel, y, x):
    px = panel[[y, x]].dropna()
    train_px = px.loc[(px.index >= train_start) & (px.index <= train_end)]
    alpha, beta = beta_ols_log(train_px[y], train_px[x])
    log_px = np.log(px)
    spread = log_px[y] - alpha - beta * log_px[x]
    spread_ret = spread.diff().fillna(0)
    return spread, spread_ret, alpha, beta

def strategy_from_signal(asset_ret, signal, cost_per_turnover):
    n = len(asset_ret)
    gross = np.zeros(n, dtype=np.float64)
    net = np.zeros(n, dtype=np.float64)
    pos = np.zeros(n, dtype=np.float64)
    for i in range(1, n):
        pos[i] = signal[i - 1] if np.isfinite(signal[i - 1]) else 0.0
        r = asset_ret[i] if np.isfinite(asset_ret[i]) else 0.0
        gross[i] = gross[i - 1] + pos[i] * r
        turn = abs(pos[i] - pos[i - 1])
        net[i] = net[i - 1] + pos[i] * r - cost_per_turnover * turn
    return gross, net, pos

def make_zscore_signal_np(x, window, direction, entry_z=1.0, clip_abs=1.0):
    mean, std = rolling_mean_std(x.astype(np.float64), int(window))
    z = (x - mean) / std
    sig = np.where(np.abs(z) >= entry_z, direction * z / entry_z, 0.0)
    sig = np.clip(sig, -clip_abs, clip_abs)
    sig[~np.isfinite(sig)] = 0.0
    return sig.astype(np.float64)

def rebuild_core_path(row):
    family = row['strategy_family']
    window = int(row['window'])
    y = row['y']
    x = row['x']
    if x == 'sector_index':
        markets = [c for c in panel_30m.columns if c != 'Date']
        # Sector paths from geometry audit are not reconstructed exactly here unless saved group returns exist.
        # Use saved per-strategy path if available; otherwise skip sector strategy.
        return None
    spread, spread_ret, alpha, beta = build_pair_spread(panel_30m, y, x)
    direction = -1.0 if 'mean_reversion' in family else 1.0
    sig = make_zscore_signal_np(spread.to_numpy(dtype=np.float64), window, direction)
    if family.startswith('inverse_'):
        sig = -sig
    gross, net, pos = strategy_from_signal(spread_ret.reindex(spread.index).fillna(0).to_numpy(dtype=np.float64), sig, float(row.get('core_cost_per_turnover', 0.00005)))
    pth = pd.DataFrame({'core_gross_pnl': gross, 'core_net_pnl': net, 'core_position': pos}, index=spread.index)
    return filter_train(pth).dropna()

In [17]:
OPTION_TYPES = {'call': 1.0, 'put': -1.0}
STD_SPANS = [22, 44, 88]
K1_SHIFT_SIGMAS = [0.0, 0.5, 1.0, 1.5]
K2_SHIFT_SIGMAS = [0.0, 0.5, 1.0]
T2_MULTIPLIERS_FOR_HL = [0.5, 1.0, 2.0]

rows = []
path_rows = []

for _, row in basket.iterrows():
    core = rebuild_core_path(row)
    if core is None or core.empty or len(core) < 500:
        rows.append({'strategy_id': row['strategy_id'], 'status': 'skipped_no_core_path'})
        continue
    pnl = core['core_net_pnl'].to_numpy(dtype=np.float64)
    base_total, base_sharpe, base_dd, base_hit, base_vol = pnl_metrics(pnl)
    adaptive_t2 = row.get('train_adaptive_T2_bars', np.nan)
    t2_grid = [MONTHLY_T_BARS]
    if np.isfinite(adaptive_t2):
        for mult in T2_MULTIPLIERS_FOR_HL:
            t2_grid.append(int(np.clip(round(float(adaptive_t2) * mult), 4, 1500)))
    t2_grid = sorted(set(int(v) for v in t2_grid if int(v) >= 4))

    for opt1_name, opt1_sign in OPTION_TYPES.items():
        for opt2_name, opt2_sign in OPTION_TYPES.items():
            for std1 in STD_SPANS:
                for std2 in STD_SPANS:
                    for k1_shift in K1_SHIFT_SIGMAS:
                        for k2_shift in K2_SHIFT_SIGMAS:
                            for t2 in t2_grid:
                                layer1, layer2, sig1, sig2, k1, k2 = two_option_bar_filter(
                                    pnl, MONTHLY_T_BARS, int(t2), int(std1), int(std2),
                                    float(opt1_sign), float(opt2_sign), float(k1_shift), float(k2_shift)
                                )
                                total, sharpe_like, max_dd, hit, vol = pnl_metrics(layer2)
                                l1_total, l1_sharpe, l1_dd, l1_hit, l1_vol = pnl_metrics(layer1)
                                turnover1 = float(np.abs(np.diff(sig1)).sum()) if len(sig1) > 1 else np.nan
                                turnover2 = float(np.abs(np.diff(sig2)).sum()) if len(sig2) > 1 else np.nan
                                rows.append({
                                    'strategy_id': row['strategy_id'],
                                    'source_group': row['source_group'],
                                    'y': row['y'], 'x': row['x'], 'strategy_family': row['strategy_family'], 'window': row['window'],
                                    'basket_bucket': row['basket_bucket'], 'geometry_class': row['geometry_class'],
                                    'core_train_total': base_total, 'core_train_sharpe_like': base_sharpe, 'core_train_max_drawdown': base_dd,
                                    'train_adaptive_T2_bars': adaptive_t2,
                                    'opt1_type': opt1_name, 'opt2_type': opt2_name,
                                    'T1_bars': MONTHLY_T_BARS, 'T2_bars': int(t2), 'std1_span': int(std1), 'std2_span': int(std2),
                                    'K1_shift_sigma': float(k1_shift), 'K2_shift_sigma': float(k2_shift),
                                    'layer1_total': l1_total, 'layer1_sharpe_like': l1_sharpe, 'layer1_max_drawdown': l1_dd,
                                    'two_option_total': total, 'two_option_sharpe_like': sharpe_like, 'two_option_max_drawdown': max_dd,
                                    'two_option_hit_rate': hit, 'two_option_step_vol': vol,
                                    'signal1_turnover': turnover1, 'signal2_turnover': turnover2,
                                    'status': 'ok',
                                })

results = pd.DataFrame(rows)
results.to_csv(OUTPUT_DIR / '02_two_option_train_grid_results.csv', index=False)
print('grid rows', len(results), 'ok', (results['status']=='ok').sum())
results.head()

grid rows 82956 ok 82944


,strategy_id,source_group,y,x,strategy_family,window,basket_bucket,geometry_class,core_train_total,core_train_sharpe_like,...,layer1_sharpe_like,layer1_max_drawdown,two_option_total,two_option_sharpe_like,two_option_max_drawdown,two_option_hit_rate,two_option_step_vol,signal1_turnover,signal2_turnover,status
0,soy_crush_and_oilseeds_RS_ZL_spread_mean_rever...,soy_crush_and_oilseeds,RS,ZL,spread_mean_reversion,44.0,positive_trend,trend_up (тренд вверх),7.5018,7.577286,...,7.305373,-0.19196,5.507826,7.137295,-0.141497,0.220356,0.001907,466.285002,649.413557,ok
1,soy_crush_and_oilseeds_RS_ZL_spread_mean_rever...,soy_crush_and_oilseeds,RS,ZL,spread_mean_reversion,44.0,positive_trend,trend_up (тренд вверх),7.5018,7.577286,...,7.305373,-0.19196,5.912250,7.102016,-0.109134,0.226526,0.002057,466.285002,412.226797,ok
2,soy_crush_and_oilseeds_RS_ZL_spread_mean_rever...,soy_crush_and_oilseeds,RS,ZL,spread_mean_reversion,44.0,positive_trend,trend_up (тренд вверх),7.5018,7.577286,...,7.305373,-0.19196,5.974668,7.098030,-0.109134,0.231289,0.002080,466.285002,369.151682,ok
3,soy_crush_and_oilseeds_RS_ZL_spread_mean_rever...,soy_crush_and_oilseeds,RS,ZL,spread_mean_reversion,44.0,positive_trend,trend_up (тренд вверх),7.5018,7.577286,...,7.305373,-0.19196,6.113913,7.144055,-0.109134,0.242527,0.002114,466.285002,239.106274,ok
4,soy_crush_and_oilseeds_RS_ZL_spread_mean_rever...,soy_crush_and_oilseeds,RS,ZL,spread_mean_reversion,44.0,positive_trend,trend_up (тренд вверх),7.5018,7.577286,...,7.305373,-0.19196,5.455173,7.117573,-0.130989,0.217587,0.001894,466.285002,651.723270,ok


In [18]:
ok = results[results['status'].eq('ok')].copy()
ok['improvement_vs_core'] = ok['two_option_total'] - ok['core_train_total']
ok['abs_two_option_total'] = ok['two_option_total'].abs()
ok['drawdown_efficiency'] = ok['two_option_total'] / ok['two_option_max_drawdown'].abs().replace(0, np.nan)

# Keep both profitable call-like and profitable put-like meta constructions.
leaderboard = ok.sort_values(
    ['two_option_total', 'two_option_sharpe_like', 'drawdown_efficiency'],
    ascending=[False, False, False],
).copy()
leaderboard.to_csv(OUTPUT_DIR / '03_two_option_train_leaderboard.csv', index=False)

by_core = leaderboard.groupby('strategy_id').head(3).reset_index(drop=True)
by_core.to_csv(OUTPUT_DIR / '04_best_three_params_per_core_strategy.csv', index=False)

summary = ok.groupby(['basket_bucket','opt1_type','opt2_type']).agg(
    n=('strategy_id','count'),
    median_total=('two_option_total','median'),
    max_total=('two_option_total','max'),
    median_sharpe=('two_option_sharpe_like','median'),
    median_T2=('T2_bars','median'),
    median_K1_shift=('K1_shift_sigma','median'),
    median_K2_shift=('K2_shift_sigma','median'),
).reset_index().sort_values(['max_total','median_total'], ascending=[False, False])
summary.to_csv(OUTPUT_DIR / '05_summary_by_bucket_and_option_combo.csv', index=False)

shift_summary = ok.groupby(['basket_bucket','opt1_type','opt2_type','K1_shift_sigma','K2_shift_sigma']).agg(
    n=('strategy_id','count'),
    median_total=('two_option_total','median'),
    max_total=('two_option_total','max'),
    median_improvement=('improvement_vs_core','median'),
    median_signal1_turnover=('signal1_turnover','median'),
    median_signal2_turnover=('signal2_turnover','median'),
).reset_index().sort_values(['max_total','median_total'], ascending=[False, False])
shift_summary.to_csv(OUTPUT_DIR / '05b_summary_by_strike_shift.csv', index=False)

leaderboard.head(30)

,strategy_id,source_group,y,x,strategy_family,window,basket_bucket,geometry_class,core_train_total,core_train_sharpe_like,...,two_option_sharpe_like,two_option_max_drawdown,two_option_hit_rate,two_option_step_vol,signal1_turnover,signal2_turnover,status,improvement_vs_core,abs_two_option_total,drawdown_efficiency
49395,soy_crush_and_oilseeds_RS_ZS_inverse_spread_me...,soy_crush_and_oilseeds,RS,ZS,inverse_spread_mean_reversion,44.0,negative_trend,trend_down (тренд вниз),-9.229028,-9.185815,...,8.781535,-0.131370,0.349130,0.002310,260.941450,190.217297,ok,17.452616,8.223588,62.598834
51123,soy_crush_and_oilseeds_RS_ZS_spread_momentum_44,soy_crush_and_oilseeds,RS,ZS,spread_momentum,44.0,negative_trend,trend_down (тренд вниз),-9.229028,-9.185815,...,8.781535,-0.131370,0.349130,0.002310,260.941450,190.217297,ok,17.452616,8.223588,62.598834
49443,soy_crush_and_oilseeds_RS_ZS_inverse_spread_me...,soy_crush_and_oilseeds,RS,ZS,inverse_spread_mean_reversion,44.0,negative_trend,trend_down (тренд вниз),-9.229028,-9.185815,...,8.781094,-0.132843,0.349059,0.002305,260.941450,190.615851,ok,17.434440,8.205412,61.767606
51171,soy_crush_and_oilseeds_RS_ZS_spread_momentum_44,soy_crush_and_oilseeds,RS,ZS,spread_momentum,44.0,negative_trend,trend_down (тренд вниз),-9.229028,-9.185815,...,8.781094,-0.132843,0.349059,0.002305,260.941450,190.615851,ok,17.434440,8.205412,61.767606
49399,soy_crush_and_oilseeds_RS_ZS_inverse_spread_me...,soy_crush_and_oilseeds,RS,ZS,inverse_spread_mean_reversion,44.0,negative_trend,trend_down (тренд вниз),-9.229028,-9.185815,...,8.760728,-0.130035,0.347841,0.002306,260.941450,198.222439,ok,17.418234,8.189206,62.976848
51127,soy_crush_and_oilseeds_RS_ZS_spread_momentum_44,soy_crush_and_oilseeds,RS,ZS,spread_momentum,44.0,negative_trend,trend_down (тренд вниз),-9.229028,-9.185815,...,8.760728,-0.130035,0.347841,0.002306,260.941450,198.222439,ok,17.418234,8.189206,62.976848
49251,soy_crush_and_oilseeds_RS_ZS_inverse_spread_me...,soy_crush_and_oilseeds,RS,ZS,inverse_spread_mean_reversion,44.0,negative_trend,trend_down (тренд вниз),-9.229028,-9.185815,...,8.731879,-0.137148,0.347443,0.002313,377.972356,288.427356,ok,17.416109,8.187080,59.695170
50979,soy_crush_and_oilseeds_RS_ZS_spread_momentum_44,soy_crush_and_oilseeds,RS,ZS,spread_momentum,44.0,negative_trend,trend_down (тренд вниз),-9.229028,-9.185815,...,8.731879,-0.137148,0.347443,0.002313,377.972356,288.427356,ok,17.416109,8.187080,59.695170
49299,soy_crush_and_oilseeds_RS_ZS_inverse_spread_me...,soy_crush_and_oilseeds,RS,ZS,inverse_spread_mean_reversion,44.0,negative_trend,trend_down (тренд вниз),-9.229028,-9.185815,...,8.745243,-0.137143,0.348193,0.002308,377.972356,196.794881,ok,17.411598,8.182570,59.664537
51027,soy_crush_and_oilseeds_RS_ZS_spread_momentum_44,soy_crush_and_oilseeds,RS,ZS,spread_momentum,44.0,negative_trend,trend_down (тренд вниз),-9.229028,-9.185815,...,8.745243,-0.137143,0.348193,0.002308,377.972356,196.794881,ok,17.411598,8.182570,59.664537


In [19]:
def downsample(df, max_points=MAX_PLOT_POINTS):
    if len(df) <= max_points:
        return df.copy()
    take = np.linspace(0, len(df)-1, max_points).astype(int)
    return df.iloc[take].copy()

def save_top_paths_and_figures(top):
    fig_rows = []
    path_rows = []
    for _, r in top.iterrows():
        core = rebuild_core_path(r)
        if core is None or core.empty:
            continue
        pnl = core['core_net_pnl'].to_numpy(dtype=np.float64)
        layer1, layer2, sig1, sig2, k1, k2 = two_option_bar_filter(
            pnl, int(r['T1_bars']), int(r['T2_bars']), int(r['std1_span']), int(r['std2_span']),
            OPTION_TYPES[r['opt1_type']], OPTION_TYPES[r['opt2_type']],
            float(r['K1_shift_sigma']), float(r['K2_shift_sigma'])
        )
        pth = pd.DataFrame({
            'Date': core.index,
            'core_net_pnl': pnl,
            'strike1': k1,
            'layer1_pnl': layer1,
            'strike2': k2,
            'two_option_pnl': layer2,
            'signal1': sig1,
            'signal2': sig2,
        })
        pth['core_minus_strike1'] = pth['core_net_pnl'] - pth['strike1']
        pth['layer1_minus_strike2'] = pth['layer1_pnl'] - pth['strike2']
        sid = r['strategy_id']
        combo = (
            f"{r['opt1_type']}_{r['opt2_type']}_T2_{int(r['T2_bars'])}"
            f"_std_{int(r['std1_span'])}_{int(r['std2_span'])}"
            f"_K_{float(r['K1_shift_sigma']):.1f}_{float(r['K2_shift_sigma']):.1f}"
        )
        safe_id = f"{sid}__{combo}".replace('/', '_')
        group_dir = FIG_DIR / str(r['basket_bucket'])
        path_group_dir = PATH_DIR / str(r['basket_bucket'])
        group_dir.mkdir(parents=True, exist_ok=True)
        path_group_dir.mkdir(parents=True, exist_ok=True)
        pds = downsample(pth)
        path_file = path_group_dir / f'{safe_id}.parquet'
        pds.to_parquet(path_file, index=False)

        fig, axes = plt.subplots(
            4, 1, figsize=(14, 10), sharex=True,
            gridspec_kw={'height_ratios': [2.0, 1.0, 2.0, 2.0]}
        )
        axes[0].plot(pds['Date'], pds['core_net_pnl'], label='core net PnL (чистый PnL базы)', linewidth=1.1)
        axes[0].step(pds['Date'], pds['strike1'], where='post', label='K1/strike1 (страйк 1)', color='black', alpha=0.65, linewidth=0.9)
        axes[0].fill_between(
            pds['Date'], pds['core_net_pnl'], pds['strike1'],
            where=(pds['core_net_pnl'] >= pds['strike1']),
            color='tab:green', alpha=0.08, step='post'
        )
        axes[0].fill_between(
            pds['Date'], pds['core_net_pnl'], pds['strike1'],
            where=(pds['core_net_pnl'] < pds['strike1']),
            color='tab:red', alpha=0.08, step='post'
        )
        axes[0].set_title(
            f"{sid} | {r['basket_bucket']} | {r['opt1_type']}/{r['opt2_type']} | "
            f"T1={int(r['T1_bars'])}, T2={int(r['T2_bars'])}, "
            f"K shifts={float(r['K1_shift_sigma']):.1f}/{float(r['K2_shift_sigma']):.1f} sigma"
        )
        axes[0].legend(loc='best')
        axes[0].set_ylabel('PnL')

        axes[1].plot(pds['Date'], pds['signal1'], label='delta1/position1 (дельта/позиция 1)', color='tab:orange', linewidth=0.9)
        axes[1].plot(pds['Date'], pds['signal2'], label='delta2/position2 (дельта/позиция 2)', color='tab:purple', linewidth=0.9)
        axes[1].axhline(0, color='black', linewidth=0.7, alpha=0.5)
        axes[1].set_ylim(-1.05, 1.05)
        axes[1].legend(loc='best')

        axes[2].plot(pds['Date'], pds['layer1_pnl'], label='layer1 PnL (PnL первого слоя)', color='tab:blue', linewidth=1.1)
        axes[2].step(pds['Date'], pds['strike2'], where='post', label='K2/strike2 (страйк 2)', color='black', alpha=0.65, linewidth=0.9)
        axes[2].fill_between(
            pds['Date'], pds['layer1_pnl'], pds['strike2'],
            where=(pds['layer1_pnl'] >= pds['strike2']),
            color='tab:green', alpha=0.08, step='post'
        )
        axes[2].fill_between(
            pds['Date'], pds['layer1_pnl'], pds['strike2'],
            where=(pds['layer1_pnl'] < pds['strike2']),
            color='tab:red', alpha=0.08, step='post'
        )
        axes[2].legend(loc='best')
        axes[2].set_ylabel('Layer1')

        axes[3].plot(pds['Date'], pds['two_option_pnl'], label='two-option PnL (PnL двух слоёв)', color='tab:green', linewidth=1.2)
        axes[3].plot(pds['Date'], pds['layer1_pnl'], label='layer1 PnL (PnL первого слоя)', color='tab:blue', linewidth=0.8, alpha=0.5)
        axes[3].plot(pds['Date'], pds['two_option_pnl'] - pds['core_net_pnl'], label='two-option minus core (разница двух слоёв и базы)', color='tab:gray', linewidth=0.8, alpha=0.8)
        axes[3].axhline(0, color='black', linewidth=0.7, alpha=0.5)
        axes[3].legend(loc='best')
        axes[3].set_ylabel('Meta PnL')
        fig.tight_layout()
        fig_file = group_dir / f'{safe_id}.png'
        fig.savefig(fig_file, dpi=110)
        plt.close(fig)
        fig_rows.append({
            'strategy_id': sid, 'basket_bucket': r['basket_bucket'], 'combo': combo,
            'K1_shift_sigma': float(r['K1_shift_sigma']), 'K2_shift_sigma': float(r['K2_shift_sigma']),
            'figure': str(fig_file), 'path_file': str(path_file), 'two_option_total': r['two_option_total'],
        })
        path_rows.append({
            'strategy_id': sid, 'basket_bucket': r['basket_bucket'], 'combo': combo,
            'K1_shift_sigma': float(r['K1_shift_sigma']), 'K2_shift_sigma': float(r['K2_shift_sigma']),
            'path_file': str(path_file), 'saved_rows': len(pds), 'full_rows': len(pth),
        })
    return pd.DataFrame(fig_rows), pd.DataFrame(path_rows)

top_for_plots = (
    leaderboard
    .sort_values(['strategy_id', 'two_option_total', 'two_option_sharpe_like'], ascending=[True, False, False])
    .groupby('strategy_id', as_index=False)
    .head(1)
    .sort_values(['basket_bucket', 'two_option_total'], ascending=[True, False])
    .head(MAX_PLOT_STRATEGIES)
)
top_for_plots.to_csv(OUTPUT_DIR / '06_best_plotted_combo_per_core_strategy.csv', index=False)
fig_index, path_index = save_top_paths_and_figures(top_for_plots)
fig_index.to_csv(OUTPUT_DIR / '06_train_top_figures.csv', index=False)
path_index.to_csv(OUTPUT_DIR / '07_train_top_path_index.csv', index=False)
print('saved figures', len(fig_index))
fig_index.head(20)

saved figures 48


,strategy_id,basket_bucket,combo,K1_shift_sigma,K2_shift_sigma,figure,path_file,two_option_total
0,soy_crush_and_oilseeds_RS_ZS_inverse_spread_me...,negative_trend,put_call_T2_1500_std_44_22_K_0.0_0.0,0.0,0.0,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,8.223588
1,soy_crush_and_oilseeds_RS_ZS_spread_momentum_44,negative_trend,put_call_T2_1500_std_44_22_K_0.0_0.0,0.0,0.0,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,8.223588
2,grain_complex_ZW_KE_inverse_spread_mean_revers...,negative_trend,put_call_T2_1500_std_22_22_K_0.0_0.0,0.0,0.0,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,7.132243
3,grain_complex_ZW_KE_spread_momentum_44,negative_trend,put_call_T2_1500_std_22_22_K_0.0_0.0,0.0,0.0,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,7.132243
4,soy_crush_and_oilseeds_RS_ZL_inverse_spread_me...,negative_trend,put_call_T2_1500_std_44_44_K_0.0_0.0,0.0,0.0,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,7.122683
5,soy_crush_and_oilseeds_RS_ZL_spread_momentum_44,negative_trend,put_call_T2_1500_std_44_44_K_0.0_0.0,0.0,0.0,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,7.122683
6,soy_crush_and_oilseeds_RS_ZS_inverse_spread_me...,negative_trend,put_call_T2_1500_std_44_22_K_0.5_0.0,0.5,0.0,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,6.519457
7,soy_crush_and_oilseeds_RS_ZS_spread_momentum_88,negative_trend,put_call_T2_1500_std_44_22_K_0.5_0.0,0.5,0.0,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,6.519457
8,soy_crush_and_oilseeds_RS_ZL_inverse_spread_me...,negative_trend,put_call_T2_1500_std_44_44_K_0.0_0.0,0.0,0.0,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,5.223226
9,soy_crush_and_oilseeds_RS_ZL_spread_momentum_88,negative_trend,put_call_T2_1500_std_44_44_K_0.0_0.0,0.0,0.0,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,5.223226


In [20]:
readme = """# two_option_train_sweep

Outputs:

- `00_protocol_splits.csv` - train/calibration/test split (разбиение на обучение/калибровку/тест).
- `01_core_strategy_basket.csv` - selected core strategies (выбранная корзина базовых стратегий).
- `02_two_option_train_grid_results.csv` - full train grid results (полный результат сетки на обучении).
- `03_two_option_train_leaderboard.csv` - sorted train leaderboard (таблица лидеров на обучении).
- `04_best_three_params_per_core_strategy.csv` - best three two-option parameter sets per core strategy (три лучших набора параметров на базовую стратегию).
- `05_summary_by_bucket_and_option_combo.csv` - aggregate summary by basket bucket and option combo (сводка по корзинам и комбинациям опционов).
- `05b_summary_by_strike_shift.csv` - aggregate summary by strike shift parameters (сводка по параметрам сдвига страйка).
- `06_best_plotted_combo_per_core_strategy.csv` - best plotted two-option combo per core strategy (лучшая нарисованная двухопционная комбинация на базовую стратегию).
- `06_train_top_figures.csv` and `figures/*/*.png` - one diagnostic chart per selected core strategy (один диагностический график на выбранную базовую стратегию).
- `07_train_top_path_index.csv` and `paths/*/*.parquet` - downsampled paths for plotted leaders (прореженные траектории нарисованных лидеров).

Protocol note: this notebook uses only train (обучение) for parameter sweep. Calibration/test (калибровка/тест) must be checked in a separate notebook to avoid look-ahead bias (смещение от знания будущего).
"""
(OUTPUT_DIR / 'README.md').write_text(readme, encoding='utf-8')
manifest = {
    'source_panel': str(PANEL_PATH.resolve()),
    'source_geometry': str(GEOMETRY_PATH.resolve()),
    'source_half_life': str(HALF_LIFE_PATH.resolve()),
    'protocol_file': str(PROTOCOL_PATH.resolve()),
    'protocol': protocol_cfg,
    'has_numba': HAS_NUMBA,
    'monthly_T_bars': MONTHLY_T_BARS,
    'std_spans': STD_SPANS,
    'k1_shift_sigmas': K1_SHIFT_SIGMAS,
    'k2_shift_sigmas': K2_SHIFT_SIGMAS,
    't2_multipliers_for_half_life': T2_MULTIPLIERS_FOR_HL,
    'saved_files': sorted([str(p.relative_to(OUTPUT_DIR)) for p in OUTPUT_DIR.rglob('*') if p.is_file()]),
}
with (OUTPUT_DIR / '00_manifest.json').open('w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2, default=str)
print('Saved to:', OUTPUT_DIR.resolve())

Saved to: ./data\two_option_train_sweep
